In [1]:
from sklearn.preprocessing import StandardScaler
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report
import tqdm
from SA_CNN_LSTM_model import MultiViewSAConvlLSTM  # Ensure this matches your actual model class

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import glob
import os
import joblib

In [2]:
# Configuration
F_DATA_PATH        = '../phonetic/front_view'  # thư mục chứa các file .npz        
L_DATA_PATH        = '../phonetic/left_view'   # thư mục chứa các file .npz
R_DATA_PATH        = '../phonetic/right_view'  # thư mục chứa các file .npz
LABEL_MAP_PATH   = '../Logs/label_map.json'  # đường dẫn đến file label_map.json
BATCH_SIZE       = 128
EPOCHS           = 100
LEARNING_RATE    = 2e-4
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


In [3]:
# 1. Load Files and Labels
with open(LABEL_MAP_PATH, 'r', encoding='utf-8') as f:
    label_map = json.load(f)

train_files = glob.glob(os.path.join(F_DATA_PATH, 'train', '**', '*.npz'), recursive=True)
val_files   = glob.glob(os.path.join(F_DATA_PATH, 'val', '**', '*.npz'), recursive=True)

# 2. Fit Scaler (Critical for Velocity vs Angles)
print("Fitting Scaler on training samples...")
scaler = StandardScaler()
sample_list = []
for f in train_files[:300]: # Sample 300 files for efficiency
    sample_list.append(np.load(f)["sequence"])
scaler.fit(np.concatenate(sample_list).reshape(-1, 64))
joblib.dump(scaler, 'vsl_scaler.pkl')
print("Scaler saved to vsl_scaler.pkl")

# 3. Dataset Class
class VSLDataset(Dataset):
    def __init__(self, front_file_list, scaler):
        self.files = front_file_list
        self.scaler = scaler

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f_path = self.files[idx]

        l_path = f_path.replace('front_view', 'left_view')
        r_path = f_path.replace('front_view', 'right_view')

        def load_npz(path):
            seq = np.load(path)["sequence"].astype(np.float32)  # (60, 64)
            seq = self.scaler.transform(seq)  # Scale features
            return torch.from_numpy(seq).view(60, 1, 8, 8)
            
        x_f = load_npz(f_path)
        x_l = load_npz(l_path)
        x_r = load_npz(r_path)

        label = int(np.load(f_path)["label"])
        
        return x_f, x_l, x_r, label

# 4. DataLoaders
train_loader = DataLoader(VSLDataset(train_files, scaler), batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True)
val_loader   = DataLoader(VSLDataset(val_files, scaler), batch_size=BATCH_SIZE, shuffle=False, num_workers=12, pin_memory=True)

Fitting Scaler on training samples...
Scaler saved to vsl_scaler.pkl


In [ ]:
def evaluate_sa_f1(model_path, val_loader, scaler, device, num_classes=400):
    # 1. Initialize the Phonetic Model
    # Ensure this class matches your original SA-CNN-LSTM architecture
    model = MultiViewSAConvlLSTM(num_classes=num_classes)
    
    # Load the saved weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    print(f"Testing Phonetic Model: {model_path}")
    
    with torch.no_grad():
        for xf_p, xl_p, xr_p, _, _, _, labels in tqdm.tqdm(val_loader, desc="Inference"):
            # Move only the phonetic views to the GPU
            xf_p, xl_p, xr_p = xf_p.to(device), xl_p.to(device), xr_p.to(device)
            labels = labels.to(device)

            # Forward pass through the phonetic branch
            # Assuming your model's forward takes (front, left, right)
            with torch.cuda.amp.autocast():
                outputs = model(xf_p, xl_p, xr_p)
                preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 2. Calculate Macro F1
    # Macro F1 is critical for VSL400 to ensure rare signs are weighted fairly
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    print(f"\n--- Phonetic Branch Results ---")
    print(f"Macro F1-Score:    {macro_f1:.4f}")
    print(f"Weighted F1-Score: {weighted_f1:.4f}")
    
    # Optional: Detailed report for all 400 Vietnamese Sign Language classes
    print(classification_report(all_labels, all_preds))
    
    return macro_f1


evaluate_sa_f1('best_vsl_model.pth', val_loader, scaler, DEVICE)

Testing Phonetic Model: best_vsl_model.pth


Inference:   0%|          | 0/23 [00:00<?, ?it/s]d:\SignLanguage\Evenv\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
